# RAG Agent (LangChain + OpenAI)

This notebook builds a simple RAG (Retrieval-Augmented Generation) agent.

- Put your documents in this same folder (`zLangChainRAG/`).
- Supported by default: `.txt`, `.md`

Env vars (in your repo `.env`):
- `OPENAI_API_KEY=...`
- `OPENAI_MODEL=gpt-4o-mini` (optional)


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.agents import create_agent

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool


In [ ]:
from getpass import getpass
os.environ['OPENAI_API_KEY'] = getpass('Voer je OpenAI API key in: ')

In [ ]:
load_dotenv(override=True)

llm = ChatOpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
)

embeddings = OpenAIEmbeddings(
    api_key=os.getenv('OPENAI_API_KEY'),
)


In [ ]:
# Load documents from this folder
# Supports: .pdf, .docx
DOCS_DIR = Path('.').resolve()   # zLangChainRAG/

loaders = [
    DirectoryLoader(
        str(DOCS_DIR),
        glob='**/*.pdf',
        loader_cls=PyPDFLoader,
        show_progress=True,
    ),
    DirectoryLoader(
        str(DOCS_DIR),
        glob='**/*.docx',
        loader_cls=Docx2txtLoader,
        show_progress=True,
    ),
]

documents = []
for loader in loaders:
    documents.extend(loader.load())

print('loaded documents:', len(documents))


In [ ]:
# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
chunks = splitter.split_documents(documents)
print('chunks:', len(chunks))


In [ ]:
# Build in-memory vector store + retriever
vectorstore = InMemoryVectorStore.from_documents(chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})


In [ ]:
# Create a Tool for retrieval (RAG)
# Some LangChain versions do not ship a helper for this,
# so we define a small tool wrapper ourselves.

@tool
def rag_search(query: str) -> str:
    """Search the local document collection for relevant context."""
    docs = retriever.invoke(query)
    if not docs:
        return 'No relevant documents found.'

    parts = []
    for i, d in enumerate(docs, start=1):
        source = d.metadata.get('source', 'unknown')
        parts.append(f"[Doc {i} | {source}]\n{d.page_content}")

    return "\n\n".join(parts)

rag_tool = rag_search


In [ ]:
system = (
    "You are a helpful assistant. "
    "Use the `rag_search` tool when the user asks questions that might be answered from the local documents. "
    "If the documents do not contain the answer, say so."
)

agent = create_agent(llm, tools=[rag_tool], system_prompt=system)


In [ ]:
question = "Who is the KPN contact person for the apartmentcomplex?"

result = agent.invoke({
    'messages': [
        {'role': 'user', 'content': question},
    ]
})

print(result['messages'][-1].content)
